In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.impute import MissingIndicator

In [2]:
np.random.seed(42)

n = 200

df = pd.DataFrame({
    "age": np.random.randint(21, 55, n),
    "experience": np.random.randint(0, 30, n),
    "salary": np.random.randint(25000, 120000, n),
    "department": np.random.choice(
        ["IT", "HR", "Finance", "Sales"], n
    )
})

df.head()

,age,experience,salary,department
0,49,19,108613,IT
1,35,23,116794,Sales
2,28,21,111672,HR
3,41,23,98847,IT
4,39,0,107750,HR


In [3]:
df_missing = df.copy()

for col in ["age", "experience", "salary", "department"]:
    mask = np.random.rand(n) < 0.15
    df_missing.loc[mask, col] = np.nan

df_missing.head(10)

,age,experience,salary,department
0,49.0,19.0,108613.0,NaN
1,35.0,23.0,NaN,Sales
2,28.0,21.0,111672.0,HR
3,41.0,23.0,NaN,IT
4,NaN,0.0,107750.0,HR
5,43.0,23.0,104634.0,Finance
6,31.0,19.0,53251.0,HR
7,31.0,10.0,50945.0,HR
8,44.0,16.0,115852.0,IT
9,44.0,7.0,57217.0,IT


In [4]:
df_missing.isnull().sum()

age           33
experience    29
salary        29
department    35
dtype: int64

In [5]:
(df_missing.isnull().mean() * 100).round(2)

age           16.5
experience    14.5
salary        14.5
department    17.5
dtype: float64

In [6]:
X_train, X_test = train_test_split(
    df_missing,
    test_size=0.20,
    random_state=42
)

X_train = X_train.copy()
X_test = X_test.copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (160, 4)
Test shape: (40, 4)


In [7]:
num_cols = ["age", "experience", "salary"]
cat_cols = ["department"]

X_train[num_cols].head()

,age,experience,salary
79,27.0,27.0,43071.0
197,39.0,17.0,NaN
38,27.0,19.0,109664.0
24,35.0,17.0,81556.0
122,45.0,7.0,63756.0


In [8]:
mean_imputer = SimpleImputer(strategy="mean")

X_train_mean = X_train.copy()
X_test_mean = X_test.copy()

X_train_mean[num_cols] = mean_imputer.fit_transform(
    X_train[num_cols]
)

X_test_mean[num_cols] = mean_imputer.transform(
    X_test[num_cols]
)

In [9]:
X_train_mean[num_cols].isnull().sum()

age           0
experience    0
salary        0
dtype: int64

In [10]:
median_imputer = SimpleImputer(strategy="median")

X_train_median = X_train.copy()
X_test_median = X_test.copy()

X_train_median[num_cols] = median_imputer.fit_transform(
    X_train[num_cols]
)

X_test_median[num_cols] = median_imputer.transform(
    X_test[num_cols]
)

In [11]:
arbitrary_imputer = SimpleImputer(
    strategy="constant",
    fill_value=-1
)

X_train_arbitrary = X_train.copy()
X_test_arbitrary = X_test.copy()

X_train_arbitrary[num_cols] = arbitrary_imputer.fit_transform(
    X_train[num_cols]
)

X_test_arbitrary[num_cols] = arbitrary_imputer.transform(
    X_test[num_cols]
)

In [12]:
end_values = (
    X_train[num_cols].mean()
    + 3 * X_train[num_cols].std()
)

X_train_eod = X_train.copy()
X_test_eod = X_test.copy()

X_train_eod[num_cols] = X_train[num_cols].fillna(end_values)
X_test_eod[num_cols] = X_test[num_cols].fillna(end_values)

In [13]:
mode_imputer = SimpleImputer(strategy="most_frequent")

X_train_mode = X_train.copy()
X_test_mode = X_test.copy()

X_train_mode[cat_cols] = mode_imputer.fit_transform(
    X_train[cat_cols]
)

X_test_mode[cat_cols] = mode_imputer.transform(
    X_test[cat_cols]
)

In [14]:
constant_imputer = SimpleImputer(
    strategy="constant",
    fill_value="Unknown"
)

X_train_constant = X_train.copy()
X_test_constant = X_test.copy()

X_train_constant[cat_cols] = constant_imputer.fit_transform(
    X_train[cat_cols]
)

X_test_constant[cat_cols] = constant_imputer.transform(
    X_test[cat_cols]
)

In [15]:
indicator = MissingIndicator(features="all")

train_indicator = pd.DataFrame(
    indicator.fit_transform(X_train),
    columns=[f"{col}_was_missing" for col in X_train.columns],
    index=X_train.index
)

test_indicator = pd.DataFrame(
    indicator.transform(X_test),
    columns=[f"{col}_was_missing" for col in X_test.columns],
    index=X_test.index
)

In [16]:
test_indicator

,age_was_missing,experience_was_missing,salary_was_missing,department_was_missing
95,False,False,True,False
15,False,True,False,False
30,False,False,False,False
158,False,False,False,False
128,False,False,False,False
115,True,True,False,False
69,True,False,False,False
170,False,False,False,False
174,False,False,False,False
45,False,False,False,False


In [17]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(
    missing_values=np.nan,
    n_neighbors=5,
    weights="distance",
    metric="nan_euclidean",
    copy=True,
    add_indicator=False,
    keep_empty_features=False
)

X_train_knn = X_train.copy()
X_test_knn = X_test.copy()

X_train_knn[num_cols] = knn_imputer.fit_transform(
    X_train[num_cols]
)

X_test_knn[num_cols] = knn_imputer.transform(
    X_test[num_cols]
)

print("Train missing values:")
print(X_train_knn[num_cols].isnull().sum())

print("\nTest missing values:")
print(X_test_knn[num_cols].isnull().sum())

Train missing values:
age           0
experience    0
salary        0
dtype: int64

Test missing values:
age           0
experience    0
salary        0
dtype: int64


In [18]:
from sklearn.linear_model import BayesianRidge

iterative_imputer = IterativeImputer(
    estimator=BayesianRidge(),
    missing_values=np.nan,
    sample_posterior=False,
    max_iter=10,
    tol=0.001,
    n_nearest_features=None,
    initial_strategy="mean",
    imputation_order="ascending",
    skip_complete=False,
    min_value=-np.inf,
    max_value=np.inf,
    verbose=0,
    random_state=42,
    add_indicator=False,
    keep_empty_features=False
)

X_train_iterative = X_train.copy()
X_test_iterative = X_test.copy()

X_train_iterative[num_cols] = iterative_imputer.fit_transform(
    X_train[num_cols]
)

X_test_iterative[num_cols] = iterative_imputer.transform(
    X_test[num_cols]
)

print("Iterations actually performed:", iterative_imputer.n_iter_)

print("\nTrain missing values:")
print(X_train_iterative[num_cols].isnull().sum())

print("\nTest missing values:")
print(X_test_iterative[num_cols].isnull().sum())

Iterations actually performed: 3

Train missing values:
age           0
experience    0
salary        0
dtype: int64

Test missing values:
age           0
experience    0
salary        0
dtype: int64


In [19]:
def random_sample_impute(train, test, columns, random_state=42):
    train_result = train.copy()
    test_result = test.copy()

    for col in columns:
        observed = train[col].dropna()

        train_missing_idx = train_result.index[
            train_result[col].isna()
        ]

        test_missing_idx = test_result.index[
            test_result[col].isna()
        ]

        train_samples = observed.sample(
            n=len(train_missing_idx),
            replace=True,
            random_state=random_state
        )

        test_samples = observed.sample(
            n=len(test_missing_idx),
            replace=True,
            random_state=random_state
        )

        train_result.loc[train_missing_idx, col] = (
            train_samples.to_numpy()
        )

        test_result.loc[test_missing_idx, col] = (
            test_samples.to_numpy()
        )

    return train_result, test_result

In [21]:
train_imputed, test_imputed = random_sample_impute(
    train=X_train,
    test=X_test,
    columns=["age", "salary"],
    random_state=42
)

In [27]:
train_imputed.isna().sum()

age            0
experience    24
salary         0
department    30
dtype: int64

In [23]:
test_imputed.isna().sum()

age           0
experience    5
salary        0
department    5
dtype: int64